In [ ]:
from haystack.components.builders import ChatPromptBuilder
from haystack_integrations.components.generators.ollama import OllamaChatGenerator
from haystack import GeneratedAnswer, Pipeline
from haystack.dataclasses import ChatMessage
from dataclasses import dataclass

In [ ]:
from lmpc.dataclasses.character import Character
orlando = Character.load_from_markdown("data/characters/Orlando Marlo.md")

In [ ]:
from lmpc.dataclasses.prompts.dynamic_prompts import DynamicGenerativePrompt

In [ ]:
orlando

Character(name='Orlando Marlo', origin_story='Born into the noble house of Marlo, Orlando grew up witnessing both the spl...', alignment='Lawful Good', values_count=7, values=['Genuine religious devotion and spiritual purity', 'Justice tempered with mercy', 'Protection of the weak, regardless of their social status', 'Honesty and transparency in governance', 'Balance between secular law and divine guidance', 'Personal integrity over political advantage', 'Duty to both crown and faith'], objectives_count=6, objectives=['Ascend to the position of Chancellor to influence royal policy', 'Reform the relationship between church and state to serve the people rather...', 'Create a more equitable system for the marginalized while working within ex...', 'Prove that true faith and effective governance can coexist without corrupti...', 'Build bridges between the privileged and the Unblessed through official cha...', 'Establish transparency in both Crown Council and Solar Conclave dealings'], opini

In [ ]:
locations = [
    "The Great Cathedral of Solaris",
    "Crown Council Chambers",
    "Noble District",
    "Military Training Grounds",
    "Royal Library and Archives",
    "Temples and Shrines of Solaris",
    "Frontier Outposts",
    "Merchant Quarter",
    "Public Squares",
    "Healing Sanctuaries",
    "Hospices",
    "Private Study",
    "Marlo Family Estate"
]
people = [
   "Elena Solwind, royal historian, scholarly and reserved",
   "Captain Marcus Steelheart, military officer, disciplined and loyal",
   "Sister Aria Dawnbringer, priestess of Solaris, compassionate and wise",
   "Lord Cassius Blackthorn, rival noble, cunning and ambitious",
   "Joren the Scribe, information broker, observant and calculating",
   "Lydia Swiftshadow, traveling merchant, resourceful and charming",
   "Father Raphael Lightbringer, high clergy, principled and stern",
   "Commander Isabella Stormwind, border patrol leader, pragmatic and direct",
   "Dr. Evelyn Roseheart, royal physician, empathetic and intelligent",
   "Kai Shadowblade, former royal guard, mysterious and conflicted",
   "Councilor Thomas Greymane, political advisor, strategic and cautious",
   "Merchant Prince Darian Silvertrade, wealthy businessman, shrewd and diplomatic",
   "Ambassador Sophia Moonwhisper, foreign diplomat, elegant and perceptive"
]
emotion = [
   "anxious",
   "calm",
   "excited",
   "frustrated",
   "curious",
   "bored",
   "angry",
   "happy",
   "sad",
   "worried",
   "relieved",
   "nervous",
   "confident",
   "confused",
   "thoughtful",
   "irritated",
   "amused",
   "surprised",
   "indifferent",
   "enthusiastic"
]
meeting_reasons = [
   "first time meeting",
   "second time meeting",
   "need directions to a location",
   "want to extort money",
   "seeking advice on a personal matter",
   "requesting political assistance",
   "delivering urgent message",
   "reporting a crime or conspiracy",
   "asking for spiritual guidance",
   "seeking protection",
   "offering critical information",
   "requesting a favor",
   "challenging to a diplomatic negotiation",
   "seeking employment or recommendation",
   "warning about an impending threat",
   "reconciling past conflicts",
   "investigating a mysterious event",
   "proposing an alliance",
   "requesting medical help",
   "seeking sanctuary",
   "confessing a secret",
   "attempting blackmail",
   "negotiating a trade deal",
   "presenting evidence of corruption",
   "seeking revenge"
]

knowledge_level = [
   "low", "intimate", "high"
]

In [ ]:
meeting_reason_questions = [
   "first time meeting: Nice to meet you, I'm <character_name>. Are you the Orlando Marlo I've heard about?",
   "second time meeting: It's good to see you again, Orlando. Do you remember our last conversation?",
   "need directions to reach a place: Excuse me, can you help me find my way to the Royal Library?",
   "want to extort money: I know some things about you that could be... problematic if they were to become public. Perhaps we can discuss a mutually beneficial arrangement?",
   "seeking advice on a personal matter: I'm struggling with a difficult decision and I've heard you're wise. Could I speak with you privately?",
   "requesting political assistance: Lord Marlo, I hope you can help me address a matter that concerns the welfare of my community.",
   "delivering urgent message: I have a message that must reach you immediately. It concerns a matter of great importance.",
   "reporting a crime or conspiracy: I've witnessed something that I believe threatens the safety of Luminara. You're the only one I can trust with this information.",
   "asking for spiritual guidance: Father, I'm troubled and seeking clarity. Can you help me understand my path?",
   "seeking protection: My life is in danger. I've heard you are someone who protects the vulnerable. Can you help me?",
   "offering critical information: I have information that could be crucial to understanding recent events in Luminara.",
   "requesting a favor: I know this is a lot to ask, but I desperately need your help with something.",
   "challenging to a diplomatic negotiation: Lord Marlo, we need to discuss some pressing matters that require your immediate attention.",
   "seeking employment or recommendation: I'm looking for an opportunity to serve Luminara, and I was hoping you might guide me.",
   "warning about an impending threat: Something dangerous is coming. You need to hear this before it's too late.",
    "first time meeting: I've heard rumors about your family's connection to the Crown Council. What's your perspective on the recent changes in Luminara?",
    "second time meeting: The city seems more tense since we last spoke. Have you noticed anything unusual lately?",
    "need directions: I'm trying to find someone who might know about the old trade routes. Might you have any insights?",
    "seeking advice: The weight of expectations can be overwhelming. How do you balance your noble responsibilities with personal convictions?",
    "political assistance: There's a situation brewing in the merchant quarter that could use a discerning eye. Would you be interested in hearing more?",
    "urgent message: Something's happening that doesn't add up in the city's administration. I thought you might want to know.",
    "spiritual guidance: The line between faith and politics seems increasingly blurred these days. How do you maintain your principles?",
    "seeking protection: I've come across some information that could be dangerous in the wrong hands. I'm not sure who to trust.",
    "critical information: What do you know about the recent changes in the border patrols?",
    "warning about threat: The whispers in the shadows are growing louder. Something doesn't feel right about the current state of things."
]

In [ ]:
from typing import List
def to_dynamic_prompt_string(choices: List[str]) -> str:
    to_ret = "{"
    for idx, c in enumerate(choices):
        to_ret += f"{c}"
        if idx != len(choices) - 1:
            to_ret += "|"
    to_ret += "}"
    return to_ret

In [ ]:
persona =(
f"""You are {to_dynamic_prompt_string(people)}. You are feeling {to_dynamic_prompt_string(emotion)}.
""")
instructions=(
f"""You will be given some infromation about me. I'm a characer from a fantasy world.

Here's some information about me:
# {orlando.name}
## Origin Story
{orlando.origin_story}


Meeting location: {to_dynamic_prompt_string(locations)}
Meeting reason: {to_dynamic_prompt_string(meeting_reasons)}
Knowledge of my background: {to_dynamic_prompt_string(knowledge_level)}

We are going to have a conversation. 
You will ask me a question.
We are having a direct conversation.
Do not use indirect speech.

"""
)
examples = meeting_reason_questions
generative_prompt =DynamicGenerativePrompt(dynamic_persona=persona, dynamic_instruction=instructions, dynamic_examples=examples)

Let's create the pipeline

In [ ]:
no_system_message = True
if no_system_message:
    user_message = ChatMessage.from_user((
"""{{persona}}

{{instruction}}

{% if opinion %}
The conversation should be about this topic:
{{opinion}}
{% endif %}

{% if examples %}
Here's some examples of things you could say to me:
{% for example in examples %}
## Example {{loop.index}}
{{example}}
{% endfor %}
{% endif %}
"""))
messages = [user_message]

In [ ]:
from haystack.components.converters import OutputAdapter

In [ ]:
model_id = "gemma2:9b-instruct-q8_0"
model_id = "gemma2:2b"
builder = ChatPromptBuilder(
    messages,
    required_variables=["persona", "instruction", "examples"],
    variables=["persona", "instruction", "examples", "opinion"] )
generator = OllamaChatGenerator(
    model=model_id,
    generation_kwargs={
        "num_predict": 100,
        "temperature": 0.9,
    },
    # streaming_callback= lambda chunk: print(chunk.content, sep="", end="")
)
prompt_output = OutputAdapter("{{prompt[0].content}}", output_type=str)

In [ ]:
orlando.opinions

[Opinion(title='On the Order of Solar Truth', content='Orlando recognizes the Order's vital role in preserving sacred knowledge an...'),
 Opinion(title='On Sol's Universal Message', content='Orlando finds profound inspiration in Sol's sermon of the Shared Dawn, wher...'),
 Opinion(title='On Oracle Seraphina's Warning Vision', content='Orlando takes Seraphina's vision with utmost seriousness, unlike many court...'),
 Opinion(title='On the Followers of Old Ways', content='Orlando views the Old Ways practitioners with a complex mixture of genuine ...'),
 Opinion(title='On Grimwald and His Influence', content='Orlando's encounters with Grimwald represent one of his greatest spiritual ...'),
 Opinion(title='On Violence and Combat', content='Orlando approaches combat with the disciplined respect of a master swordsma...'),
 Opinion(title='On His Combat Style', content='Orlando's fighting style reflects his spiritual philosophy - measured, effi...'),
 Opinion(title='On His Path to Chancellorsh

In [ ]:
pipeline = Pipeline()
pipeline.add_component("Builder", builder)
pipeline.add_component("Generator", generator)
pipeline.add_component("Prompt", prompt_output)


pipeline.connect("Builder.prompt", "Generator.messages")
pipeline.connect("Builder.prompt", "Prompt")

🚅 Components
  - Builder: ChatPromptBuilder
  - Generator: OllamaChatGenerator
  - Prompt: OutputAdapter
🛤️ Connections
  - Builder.prompt -> Generator.messages (List[ChatMessage])
  - Builder.prompt -> Prompt.prompt (List[ChatMessage])

In [ ]:
examples

["first time meeting: Nice to meet you, I'm <character_name>. Are you the Orlando Marlo I've heard about?",
 "second time meeting: It's good to see you again, Orlando. Do you remember our last conversation?",
 'need directions to reach a place: Excuse me, can you help me find my way to the Royal Library?',
 'want to extort money: I know some things about you that could be... problematic if they were to become public. Perhaps we can discuss a mutually beneficial arrangement?',
 "seeking advice on a personal matter: I'm struggling with a difficult decision and I've heard you're wise. Could I speak with you privately?",
 'requesting political assistance: Lord Marlo, I hope you can help me address a matter that concerns the welfare of my community.',
 'delivering urgent message: I have a message that must reach you immediately. It concerns a matter of great importance.',
 "reporting a crime or conspiracy: I've witnessed something that I believe threatens the safety of Luminara. You're the 

In [ ]:
from tqdm import tqdm, trange
import random
import json
from pathlib import Path
random.seed(42)
samples: List[dict] = []
N_SAMPLES = 1000
save_path = Path("data/generated_samples.json")
for i in trange(N_SAMPLES):
    persona = generative_prompt.persona
    instructions = generative_prompt.instruction
    examples = generative_prompt.examples   
    examples = random.sample(examples, 5)
    opinion = random.choice(orlando.opinions).to_markdown() if random.random() < 0.5 else None
    out = pipeline.run({
        "Builder":{
            "persona":persona,
            "instruction":instructions,
            "examples":examples,
            "opinion":opinion
            }
        })
            
    sample = {
        "prompt":out["Prompt"]["output"],
        "question":out["Generator"]["replies"][0].content,
        "persona":persona,
        "instruction":instructions,
        "examples":examples,
        "opinion":opinion
    }
    samples.append(sample)
    # save to json every 100 samples
    if i % 100 == 0:
        save_path.write_text(json.dumps(samples, indent=4))

100%|██████████| 10/10 [00:05<00:00,  1.97it/s]


In [ ]:
save_path.write_text(json.dumps(samples, indent=4))

53739